# Learning an Image Classification Model from Scratch (PyTorch)

This is the PyTorch version of the Keras [course notebook](course_notebook.ipynb). The data loading and exploration sections are identical — only the model building, training, and evaluation sections differ.

## Technical Preliminaries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## Load and explore the dataset

We use Keras just for the dataset download (it's the most convenient source), then work entirely in PyTorch.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [ ]:
print(x_train.shape, y_train.shape)

There are 60,000 images in the training set, each of which is a 28x28 matrix.

In [ ]:
print(x_test.shape, y_test.shape)

The remaining 10,000 images are in the test set.

In [ ]:
labels = ["T-shirt/top",
          "Trouser",
          "Pullover",
          "Dress",
          "Coat",
          "Sandal",
          "Shirt",
          "Sneaker",
          "Bag",
          "Ankle boot"]

In [ ]:
labels[y_train[0]]

Let's look at the first 25 images.

In [ ]:
fig, ax = plt.subplots(5, 5, figsize=(30, 10))
for i in range(5):
    for j in range(5):
        idx = i * 5 + j
        ax[i, j].imshow(x_train[idx], cmap='gray')
        ax[i, j].set_title(labels[y_train[idx]])
        ax[i, j].axis('off')
plt.tight_layout()
plt.show()

## Data Prep

NNs learn best when each independent variable is in a small range. So, let's normalize the pixel values from [0, 255] to [0, 1].

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

### Convert to PyTorch tensors and create DataLoader

In [ ]:
# Split training data into train and validation (80/20)
n_val = int(len(x_train) * 0.2)
x_val_t = torch.tensor(x_train[:n_val], dtype=torch.float32)
y_val_t = torch.tensor(y_train[:n_val], dtype=torch.long)
x_tr_t = torch.tensor(x_train[n_val:], dtype=torch.float32)
y_tr_t = torch.tensor(y_train[n_val:], dtype=torch.long)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(x_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)

## Build a model

### Define model in PyTorch

We'll start with a simple one-hidden-layer model, mirroring the Keras notebook's baseline:
- Flatten the 28x28 image into a 784-long vector
- Hidden layer with 256 units and ReLU activation
- Output layer with 10 units (one per class)

Note: In PyTorch, `CrossEntropyLoss` applies softmax internally, so we don't add a softmax layer.

In [ ]:
class FashionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),   # Hidden layer: 784 -> 256
            nn.ReLU(),
            nn.Linear(256, 10),    # Output layer: 256 -> 10
        )

    def forward(self, x):
        return self.net(x)

model = FashionNet()

In [ ]:
print(model)

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")

Let's hand-calculate the number of parameters to verify.

In [ ]:
(784 * 256 + 256) + (256 * 10 + 10)

### Set optimization parameters

In PyTorch, we set up the loss function and optimizer as separate objects.

* **Loss function**: `nn.CrossEntropyLoss()` — combines log-softmax and NLL loss. Equivalent to Keras' `sparse_categorical_crossentropy`.
* **Optimizer**: `torch.optim.Adam` — the same Adam optimizer used in the Keras notebook.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Train the model

Unlike Keras where `model.fit()` handles the entire training loop, in PyTorch we write the training loop explicitly. This gives us full control over what happens at each step:

1. **Forward pass**: compute predictions
2. **Compute loss**: compare predictions to targets
3. **Backward pass**: compute gradients
4. **Update weights**: optimizer takes a step

In [ ]:
num_epochs = 20

history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0

    for xb, yb in train_loader:
        optimizer.zero_grad()           # reset gradients
        logits = model(xb)              # forward pass
        loss = criterion(logits, yb)    # compute loss
        loss.backward()                 # backward pass
        optimizer.step()                # update weights

        epoch_loss += loss.item() * len(xb)
        epoch_correct += (logits.argmax(1) == yb).sum().item()
        epoch_total += len(xb)

    train_loss = epoch_loss / epoch_total
    train_acc = epoch_correct / epoch_total

    # --- Validation phase ---
    model.eval()
    with torch.no_grad():
        val_logits = model(x_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_acc = (val_logits.argmax(1) == y_val_t).float().mean().item()

    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - "
          f"val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

Let's plot the loss and accuracy curves to see if **overfitting** is going on (i.e., the model has started memorizing the training data).

In [ ]:
epochs_range = range(1, num_epochs + 1)
plt.plot(epochs_range, history["loss"], "bo", label="Training loss", markersize=2)
plt.plot(epochs_range, history["val_loss"], "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()
plt.plot(epochs_range, history["accuracy"], "bo", label="Training acc", markersize=2)
plt.plot(epochs_range, history["val_accuracy"], "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Evaluate the model

Let's see how well the model does on the test set.

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(x_test_t)
    test_loss = criterion(test_logits, y_test_t).item()
    test_acc = (test_logits.argmax(1) == y_test_t).float().mean().item()

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## PyTorch vs Keras: key differences

| | Keras | PyTorch |
|---|---|---|
| **Model definition** | Sequential or Functional API | Subclass `nn.Module` |
| **Training** | `model.fit()` (one line) | Explicit loop (forward, loss, backward, step) |
| **Evaluation** | `model.evaluate()` | Manual with `torch.no_grad()` |
| **Optimizer setup** | `model.compile()` | Separate `torch.optim` object |
| **Train/eval mode** | Automatic | Must call `model.train()` / `model.eval()` |
| **Loss for multi-class** | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss()` (includes softmax) |

PyTorch requires more code but gives you full visibility into the training process.